In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append('../src')

from data_processing import DataLoader, DataCleaner
from visualization.plots import plot_keyword_frequency

In [ ]:
# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Plot settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [4]:
# Initialize loader
loader = DataLoader('../config.yaml')

# Load raw data
df = loader.load_raw_data('../data/raw/consumer_complaints.csv')

print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

df.head()

INFO:data_processing.data_loader:Loading data from ../data/raw/consumer_complaints.csv
INFO:data_processing.data_loader:Successfully loaded 1058741 records


Loaded: 1058741 rows, 18 columns


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,12/28/25,Debt collection,Telecommunications debt,Attempts to collect debt not owed,Debt was result of identity theft,"I do not recognize the aforementioned accounts, collections, or hard inquiries as reported. The ...",Company has responded to the consumer and the CFPB and chooses not to provide a public response,"I.C. System, Inc.",IN,46201,NaN,Consent provided,Web,12/28/25,Closed with explanation,Yes,NaN,18327208
1,09/26/25,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,"I am writing pursuant to the Fair Credit Reporting Act ( FCRA ), particularly Section 605B [ 15 ...",NaN,"EQUIFAX, INC.",LA,71457,NaN,Consent provided,Web,09/26/25,Closed with non-monetary relief,Yes,NaN,16207045
2,03/15/25,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,I am writing to formally dispute several unauthorized transactions on my account that occurred a...,NaN,TD BANK US HOLDING COMPANY,MA,01803,NaN,Consent provided,Web,03/15/25,Closed with explanation,Yes,NaN,12493914
3,05/03/25,Debt collection,Other debt,Attempts to collect debt not owed,Debt was result of identity theft,"I am sending this asking for your help to file a formal complaint against XXXX, XXXX XXXX XXXX r...",NaN,"Portfolio Recovery Associates, LLC",TX,77373,NaN,Consent provided,Web,05/03/25,Closed with non-monetary relief,Yes,NaN,13317604
4,04/29/25,Credit reporting or other personal consumer reports,Credit reporting,Improper use of your report,Reporting company used your report improperly,"Upon reviewing my consumer reports, l discovered some instances of late payments on an account f...",Company has responded to the consumer and the CFPB and chooses not to provide a public response,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",NJ,08016,NaN,Consent provided,Web,04/29/25,Closed with non-monetary relief,Yes,NaN,13232751


In [5]:
df_original = df.copy()

# Handle missing complaint narratives
text_col = 'Consumer complaint narrative'

print(f"\nMissing narratives: {df[text_col].isna().sum()}")

# Remove rows without complaint text
df = df[df[text_col].notna()].copy()

print(f"After removing nulls: {df.shape}")


Missing narratives: 0
After removing nulls: (1058741, 18)


In [6]:
# Clean complaint text
def clean_complaint_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # 1. Remove masked data (XXXX patterns)
    text = re.sub(r'\bX{2,}\b', '', text) # XXXX -> space
    text = re.sub(r'\{X{2,}\}', '', text) # {XXXX} -> space
    
    # 2. Remove incomplete references [15 ...
    text = re.sub(r'\[\s*\d+\s*\.{2,}\s*\]?', '', text)
    
    # 3. Clean excessive whitespace (but keep single spaces)
    text = ' '.join(text.split())
    
    # 4. Strip leading/trailing whitespace
    text = text.strip()
    
    return text

In [8]:
import re

In [9]:
# Apply cleaning
df['complaint_clean'] = df[text_col].apply(clean_complaint_text)

# Show example
print("BEFORE CLEANING:")
print(df[text_col].iloc[3][:300])
print("AFTER CLEANING:")
print(df['complaint_clean'].iloc[3][:300])

BEFORE CLEANING:
I am sending this asking for your help to file a formal complaint against XXXX, XXXX XXXX XXXX regarding an account I have with them XXXXXXXX XXXX XXXX I believe that this account needs to be removed from my credit report due to error, fraud, and be remove. Here are the details of my complaint : XXX
AFTER CLEANING:
I am sending this asking for your help to file a formal complaint against , regarding an account I have with them I believe that this account needs to be removed from my credit report due to error, fraud, and be remove. Here are the details of my complaint : ( Original Creditor : ) Balance : {$1700.


In [10]:
# Create text features
df['char_count'] = df['complaint_clean'].str.len()
df['word_count'] = df['complaint_clean'].str.split().str.len()
df['sentence_count'] = df['complaint_clean'].str.count(r'[.!?]+')

# Flag truncated text
df['is_truncated'] = df[text_col].str.endswith('...', na=False)

# Summary stats
print("\nText Statistics:")
print(df[['char_count', 'word_count', 'sentence_count']].describe())


Text Statistics:
         char_count    word_count  sentence_count
count  1.058741e+06  1.058741e+06    1.058741e+06
mean   9.406174e+02  1.584360e+02    9.891899e+00
std    1.121299e+03  1.877009e+02    1.326668e+01
min    0.000000e+00  0.000000e+00    0.000000e+00
25%    3.130000e+02  5.500000e+01    3.000000e+00
50%    6.420000e+02  1.050000e+02    6.000000e+00
75%    1.168000e+03  1.990000e+02    1.200000e+01
max    3.591600e+04  5.905000e+03    1.084000e+03


In [11]:
# Filter short complaints
min_words = 10

print(f"\nComplaints with < {min_words} words: {(df['word_count'] < min_words).sum()}")

df = df[df['word_count'] >= min_words].copy()

print(f"After filtering: {df.shape}")


Complaints with < 10 words: 9295
After filtering: (1049446, 23)


In [12]:
# Clean date fields
df['Date received'] = pd.to_datetime(df['Date received'], format='%m/%d/%y', errors='coerce')
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'], format='%m/%d/%y', errors='coerce')

# Extract date features
df['year'] = df['Date received'].dt.year
df['month'] = df['Date received'].dt.month
df['month_name'] = df['Date received'].dt.month_name()
df['day_of_week'] = df['Date received'].dt.day_name()

In [13]:
# Clean categorical fields
# Product - Title case
df['Product'] = df['Product'].str.strip().str.title()

# Company - Uppercase
df['Company'] = df['Company'].str.upper().str.strip()

# State - Uppercase, 2 letters
df['State'] = df['State'].str.upper().str.strip()

# Fill missing values
df['Company public response'] = df['Company public response'].fillna('No public response')
df['ZIP code'] = df['ZIP code'].fillna('00000')
df['Tags'] = df['Tags'].fillna('No tags')


In [14]:
# Select columns to keep
columns_to_keep = [
    'Date received',
    'year',
    'month',
    'month_name',
    'day_of_week',
    'Product',
    'Sub-product',
    'Issue',
    'Sub-issue',
    'Company',
    'State',
    'Company response to consumer',
    'Timely response?',
    'Consumer disputed?',
    'complaint_clean',
    'char_count',
    'word_count',
    'sentence_count',
    'is_truncated'
]

df_processed = df[columns_to_keep].copy()

print(f"\nFinal processed dataset: {df_processed.shape}")


Final processed dataset: (1049446, 19)


In [15]:
# Save processed data 
df_processed.to_csv('../data/processed/consumer_complaints_processed.csv', index=False)

In [16]:
summary = {
    'Original records': len(df_original),
    'After removing null narratives': len(df),
    'After filtering short complaints': len(df_processed),
    'Records removed': len(df_original) - len(df_processed),
    'Removal rate': f"{((len(df_original) - len(df_processed)) / len(df_original) * 100):.2f}%",
    '': '',
    'Average characters': f"{df_processed['char_count'].mean():.0f}",
    'Average words': f"{df_processed['word_count'].mean():.0f}",
    'Average sentences': f"{df_processed['sentence_count'].mean():.1f}",
    ' ': '',
    'Truncated complaints': df_processed['is_truncated'].sum(),
    'Unique products': df_processed['Product'].nunique(),
    'Unique companies': df_processed['Company'].nunique(),
    'Date range': f"{df_processed['Date received'].min()} to {df_processed['Date received'].max()}"
}

for key, value in summary.items():
    print(f"{key:.<40} {value}")

Original records........................ 1058741
After removing null narratives.......... 1049446
After filtering short complaints........ 1049446
Records removed......................... 9295
Removal rate............................ 0.88%
........................................ 
Average characters...................... 948
Average words........................... 160
Average sentences....................... 10.0
 ....................................... 
Truncated complaints.................... 610
Unique products......................... 11
Unique companies........................ 3075
Date range.............................. 2025-01-20 00:00:00 to 2026-01-14 00:00:00
